# INFO422 Data Science Project — Data Acquisition & Preprocessing

> **Group 23** | LZU INFO422 — Influencing Factors behind Video Game Sales
>
> **Author:** Zhong Rui (Machine Learning Engineer)
> **Module:** M2 & M3 — Data Acquisition & Preprocessing (Weeks 3–4)
> **Framework:** CRISP-DM

## Notebook Structure

| Module | Content | Status |
|--------|---------|--------|
| **M2** | Data Acquisition | ✅ Implemented |
| **M3** | Data Preprocessing (Cleaning, Reduction, Transformation) | ✅ Implemented |
| **M4** | Exploratory Data Analysis (EDA) | 🏗️ Framework |
| **M5** | Correlation Analysis | 🏗️ Framework |
| **M6** | Model Assistance & Verification | 🏗️ Framework |

---

## Research Questions

1. **Q1 — Platform-Genre Sales Advantage:** How do total sales volumes vary across platform-genre combinations?
2. **Q2 — Genre-Regional Impact:** Do different genres impact sales differently across regions (NA, JP, PAL, Other)?
3. **Q3 — Brand Effect:** Does developer/publisher brand significantly impact total and regional sales?

## 0. Environment Setup

In [ ]:
# ============================================================
# 0. Environment Setup & Imports
# ============================================================

import os
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical tests
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, shapiro

# Machine learning
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

# Gradient boosting (optional — install via pip if needed)
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not installed. Run: pip install xgboost")

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("⚠️  LightGBM not installed. Run: pip install lightgbm")

# Settings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Environment ready." )
print(f"📅 Execution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. M2 — Data Acquisition

In [ ]:
# ============================================================
# 1. Data Acquisition
# ============================================================

DATA_PATH = "vgchartz-2024.csv"

print("━" * 60)
print("M2 — DATA ACQUISITION")
print("━" * 60)

# Load raw dataset
df_raw = pd.read_csv(DATA_PATH)

print(f"📁 Dataset loaded: {DATA_PATH}")
print(f"📊 Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"💾 Memory: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n" + "━" * 60)
print("Column Overview:")
print("━" * 60)

col_info = pd.DataFrame({
    "Column": df_raw.columns,
    "Non-Null": df_raw.notna().sum().values,
    "Null": df_raw.isna().sum().values,
    "Dtype": df_raw.dtypes.values,
    "Unique": [df_raw[col].nunique() for col in df_raw.columns]
})
print(col_info.to_string(index=False))

# Store raw shape for comparison
RAW_SHAPE = df_raw.shape
print(f"\n✅ Raw data acquisition complete. {RAW_SHAPE[0]:,} records acquired.")

## 2. M3 — Data Preprocessing

Preprocessing follows the CRISP-DM framework across three phases.
The goal is to produce a clean, analysis-ready, feature-rich dataset.

**Full Pipeline:**

```
Raw Data (64,016 × 14)
    │
    ▼
Phase 1: DATA CLEANING
    ├── 1.1 Column Selection ──────────► (64,016 × 11)
    ├── 1.2 Categorical Imputation ─────► ("Unknown" for publisher/developer)
    ├── 1.3 Regional Sales Imputation ──► (NaN → 0)
    ├── 1.4 Critic Score Imputation ────► (median)
    ├── 1.5 Sales Filtering ────────────► (17,570 × 11)
    └── 1.6 Outlier Flagging (IQR) ─────► (5 outlier flag columns)
    │
    ▼
Phase 2: DATA REDUCTION
    ├── 2.1 Stratified Sampling ────────► (genre-stratified subset)
    ├── 2.2 K-Means Clustering ─────────► (K=4 sales + platform clusters)
    └── 2.3 Console×Genre Aggregation ──► (min 3 games per combo)
    │
    ▼
Phase 3: DATA TRANSFORMATION
    ├── 3.1 Text Normalization ─────────► (uppercase, deduplication)
    ├── 3.2 Log Transformation ─────────► (log_sales)
    ├── 3.3 Normalization ──────────────► (Min-Max sales, Z-Score critic)
    ├── 3.4 Release Date Discretization ─► (equal-frequency 5 bins)
    └── 3.5 Derived Features ───────────► (ratios, brand proxies, flags)
    │
    ▼
Final Dataset (~22 columns)
```

**Phases:**

1. **Data Cleaning** — Handle missing values, filter invalid records, flag outliers, select relevant columns.
2. **Data Reduction** — Stratified sampling, K-Means clustering, and aggregation to identify patterns.
3. **Data Transformation** — Normalise text, log-transform target, scale features, discretize dates, engineer derived features.


### 2.1 Data Cleaning

In [ ]:
# ============================================================
# 2.1 Data Cleaning
# ============================================================

print("\n" + "━" * 60)
print("M3 — PHASE 1: DATA CLEANING")
print("━" * 60)

df = df_raw.copy()
cleaning_log = []

def log_step(step, action, justification, before, after):
    """Log a preprocessing step."""
    cleaning_log.append({
        "step": step,
        "action": action,
        "justification": justification,
        "rows_before": before,
        "rows_after": after,
        "rows_removed": before - after if after is not None else None
    })
    print(f"\n🔧 Step {step}: {action}")
    print(f"   Justification: {justification}")
    if after is not None:
        print(f"   Rows: {before:,} → {after:,} (Δ {before - after:,})")

# ─────────────────────────────────────────────────────────────
# 2.1.1 Column Selection
# ─────────────────────────────────────────────────────────────

analysis_cols = [
    'console', 'genre', 'publisher', 'developer', 'critic_score',
    'total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales',
    'release_date'  # Retained for temporal feature engineering
]

dropped_cols = [col for col in df.columns if col not in analysis_cols]
df = df[analysis_cols].copy()

log_step(
    "1.1",
    f"Column selection — retained {len(analysis_cols)} columns, dropped {len(dropped_cols)}",
    "Removed img (image URLs), title (unique ID), last_update (metadata). Retained release_date for temporal feature engineering.",
    RAW_SHAPE[0], df.shape[0]
)
print(f"   Retained: {analysis_cols}")
print(f"   Dropped:  {dropped_cols}")

# ─────────────────────────────────────────────────────────────
# 2.1.2 Handle Missing Values in Categorical Fields
# ─────────────────────────────────────────────────────────────

cat_cols = ['console', 'genre', 'publisher', 'developer']
cat_missing_before = df[cat_cols].isna().sum().sum()

for col in cat_cols:
    missing_count = df[col].isna().sum()
    if missing_count > 0:
        df[col] = df[col].fillna('Unknown')
        print(f"   {col}: filled {missing_count} missing values with 'Unknown'")

cat_missing_after = df[cat_cols].isna().sum().sum()
log_step(
    "1.2",
    "Categorical missing value imputation — filled with 'Unknown'",
    "Preserves information by creating an explicit 'Unknown' category rather than deleting records. Prevents data loss while flagging incomplete records.",
    RAW_SHAPE[0], df.shape[0]
)
print(f"   Total categorical missing values: {cat_missing_before} → {cat_missing_after}")

# ─────────────────────────────────────────────────────────────
# 2.1.3 Handle Missing Values in Regional Sales
# ─────────────────────────────────────────────────────────────

regional_cols = ['na_sales', 'jp_sales', 'pal_sales', 'other_sales']
regional_missing = df[regional_cols].isna().sum()

print("\n   Regional sales missing values before handling:")
for col in regional_cols:
    print(f"     {col}: {regional_missing[col]:,} ({regional_missing[col]/len(df)*100:.2f}%)")

# Fill regional missing values with 0
for col in regional_cols:
    df[col] = df[col].fillna(0)

log_step(
    "1.3",
    "Regional sales missing values — filled with 0",
    "Missing regional sales indicate no reported sales in that region. Filling with 0 is conservative and prevents inflating regional averages. total_sales remains the primary target variable.",
    RAW_SHAPE[0], df.shape[0]
)

# ─────────────────────────────────────────────────────────────
# 2.1.4 Handle Missing Values in Critic Score
# ─────────────────────────────────────────────────────────────

critic_missing = df['critic_score'].isna().sum()
critic_median = df['critic_score'].median()

if critic_missing > 0:
    df['critic_score'] = df['critic_score'].fillna(critic_median)
    print(f"\n   critic_score: filled {critic_missing:,} missing values with median ({critic_median:.2f})")
else:
    print(f"\n   critic_score: no missing values")

log_step(
    "1.4",
    f"Critic score imputation — median ({critic_median:.2f})",
    "Median is robust to outliers compared to mean. Critic score distribution is approximately symmetric, making median a reliable central tendency measure.",
    RAW_SHAPE[0], df.shape[0]
)

# ─────────────────────────────────────────────────────────────
# 2.1.5 Handle Missing Release Date
# ─────────────────────────────────────────────────────────────

release_missing = df['release_date'].isna().sum()
if release_missing > 0:
    print(f"\n   release_date: {release_missing:,} missing values detected.")
    print("   Will be handled in transformation (discretization) phase.")
else:
    print(f"\n   release_date: no missing values")

# ─────────────────────────────────────────────────────────────
# 2.1.6 Filter Invalid Sales Records
# ─────────────────────────────────────────────────────────────

before_filter = len(df)
invalid_sales = (df['total_sales'] <= 0).sum()

if invalid_sales > 0:
    df = df[df['total_sales'] > 0].copy()
    print(f"\n   Removed {invalid_sales:,} records with total_sales ≤ 0")
else:
    print(f"\n   No invalid sales records found (total_sales ≤ 0)")

log_step(
    "1.5",
    "Sales filtering — removed total_sales ≤ 0",
    "Zero or negative sales are data errors, placeholders, or unreleased games. They lack analytical meaning and would distort statistical summaries.",
    before_filter, len(df)
)

# ─────────────────────────────────────────────────────────────
# 2.1.7 Outlier Detection & Flagging (IQR Method)
# ─────────────────────────────────────────────────────────────

print("\n   Outlier detection using IQR method (threshold: 1.5 × IQR):")

sales_cols = ['total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales']
outlier_flags = {}

for col in sales_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outlier_flags[f'{col}_outlier'] = ((df[col] < lower_bound) | (df[col] > upper_bound)).astype(int)
    
    print(f"     {col}: {outliers:,} outliers ({outliers/len(df)*100:.2f}%) — bounds: [{lower_bound:.3f}, {upper_bound:.3f}]")

# Add outlier flag columns
for flag_col, flag_values in outlier_flags.items():
    df[flag_col] = flag_values

log_step(
    "1.6",
    "Outlier flagging — IQR method (1.5 × IQR)",
    "Extreme sales values are genuine (blockbuster hits), not errors. Flagging rather than removing preserves information while enabling outlier-aware analysis. Log transformation further dampens outlier impact.",
    len(df), len(df)
)

print(f"\n✅ Data cleaning complete. Dataset shape: {df.shape}")


### 2.2 Data Reduction

In [ ]:
# ============================================================
# 2.2 Data Reduction
# ============================================================

print("\n" + "━" * 60)
print("M3 — PHASE 2: DATA REDUCTION")
print("━" * 60)

reduction_log = []

def log_reduction(step, action, justification, shape_before, shape_after):
    reduction_log.append({
        "step": step,
        "action": action,
        "justification": justification,
        "shape_before": shape_before,
        "shape_after": shape_after
    })
    print(f"\n🔧 {step}: {action}")
    print(f"   Justification: {justification}")
    print(f"   Shape: {shape_before} → {shape_after}")

shape_before_reduction = df.shape

# ─────────────────────────────────────────────────────────────
# 2.2.1 Stratified Sampling by Genre
# ─────────────────────────────────────────────────────────────

# Use 50% stratified sample to reduce computation while preserving distribution
SAMPLE_FRAC = 0.5

# Ensure minimum representation for rare genres
genre_counts = df['genre'].value_counts()
min_samples_per_genre = 5

sampled_indices = []
for genre_name, group in df.groupby('genre'):
    n_samples = max(min_samples_per_genre, int(len(group) * SAMPLE_FRAC))
    if len(group) >= min_samples_per_genre:
        sampled_indices.extend(group.sample(n=n_samples, random_state=RANDOM_STATE).index.tolist())
    else:
        sampled_indices.extend(group.index.tolist())
df_sampled = df.loc[sampled_indices].reset_index(drop=True)

log_reduction(
    "2.1",
    f"Stratified sampling by genre ({SAMPLE_FRAC*100:.0f}%)",
    "Maintains genre proportions while reducing dataset size for faster prototyping. Rare genres preserved via minimum sample threshold.",
    shape_before_reduction, df_sampled.shape
)

# Show genre distribution preservation
print("\n   Genre distribution comparison (top 5):")
genre_comparison = pd.DataFrame({
    "Original": df['genre'].value_counts(normalize=True).head(),
    "Sampled": df_sampled['genre'].value_counts(normalize=True).head()
})
genre_comparison['diff'] = genre_comparison['Sampled'] - genre_comparison['Original']
print(genre_comparison.to_string())

# Use sampled data for subsequent reduction steps
df_reduced = df_sampled.copy()

# ─────────────────────────────────────────────────────────────
# 2.2.2 K-Means Clustering (Sales + Platform Pattern)
# ─────────────────────────────────────────────────────────────

# Encode platform for clustering
le_console = LabelEncoder()
console_encoded = le_console.fit_transform(df_reduced['console'])

# Features for clustering: total_sales + encoded platform
cluster_features = pd.DataFrame({
    'total_sales': df_reduced['total_sales'],
    'console_encoded': console_encoded,
    'critic_score': df_reduced['critic_score']
})

# Scale features for K-Means
scaler = StandardScaler()
cluster_features_scaled = scaler.fit_transform(cluster_features)

# Determine optimal K using elbow method (simplified)
k_candidates = range(2, 8)
inertias = []
for k in k_candidates:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans.fit(cluster_features_scaled)
    inertias.append(kmeans.inertia_)

# Select K=4 (balanced between granularity and interpretability)
OPTIMAL_K = 4
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=10)
df_reduced['sales_cluster'] = kmeans.fit_predict(cluster_features_scaled)

# Analyze clusters
cluster_summary = df_reduced.groupby('sales_cluster').agg({
    'total_sales': ['mean', 'median', 'count'],
    'critic_score': 'mean',
    'console': lambda x: x.mode()[0] if not x.mode().empty else 'N/A'
}).round(2)

log_reduction(
    "2.2",
    f"K-Means clustering (K={OPTIMAL_K}) on sales + platform + critic_score",
    "Identifies natural groupings of games by commercial performance and platform type. Cluster labels serve as engineered features for downstream modeling.",
    df_reduced.shape, df_reduced.shape
)

print("\n   Cluster Summary:")
print(cluster_summary.to_string())

# ─────────────────────────────────────────────────────────────
# 2.2.3 Aggregation by Console + Genre
# ─────────────────────────────────────────────────────────────

# Create aggregated view: average sales per console-genre combination
console_genre_agg = df_reduced.groupby(['console', 'genre']).agg({
    'total_sales': ['mean', 'median', 'std', 'count'],
    'na_sales': 'mean',
    'jp_sales': 'mean',
    'pal_sales': 'mean',
    'other_sales': 'mean',
    'critic_score': 'mean'
}).round(3)

# Flatten column names
console_genre_agg.columns = ['_'.join(col).strip() for col in console_genre_agg.columns.values]
console_genre_agg = console_genre_agg.reset_index()

# Filter combinations with at least 3 games for statistical reliability
console_genre_agg = console_genre_agg[console_genre_agg['total_sales_count'] >= 3]

log_reduction(
    "2.3",
    "Aggregation by console + genre (min 3 games per combo)",
    "Summarizes sales patterns at the platform-genre level, directly addressing Q1. Enables identification of high-performing combinations for strategic recommendations.",
    df_reduced.shape, console_genre_agg.shape
)

print(f"\n   Console-Genre combinations (≥3 games): {len(console_genre_agg)}")
print(f"   Top 5 by average total sales:")
top_combos = console_genre_agg.nlargest(5, 'total_sales_mean')[
    ['console', 'genre', 'total_sales_mean', 'total_sales_count']
]
print(top_combos.to_string(index=False))

print(f"\n✅ Data reduction complete.")



### 2.3 Data Transformation & Discretization

In [ ]:
# ============================================================
# 2.3 Data Transformation & Discretization
# ============================================================

print("\n" + "━" * 60)
print("M3 — PHASE 3: DATA TRANSFORMATION & DISCRETIZATION")
print("━" * 60)

transformation_log = []

def log_transform(step, action, justification):
    transformation_log.append({
        "step": step,
        "action": action,
        "justification": justification
    })
    print(f"\n🔧 {step}: {action}")
    print(f"   Justification: {justification}")

df_transformed = df_reduced.copy()

# ─────────────────────────────────────────────────────────────
# 2.3.1 Text Normalization (Console, Genre, Publisher, Developer)
# ─────────────────────────────────────────────────────────────

text_cols = ['publisher', 'developer']
for col in text_cols:
    df_transformed[col] = df_transformed[col].str.strip().str.upper()

# Also normalize console and genre
df_transformed['console'] = df_transformed['console'].str.strip().str.upper()
df_transformed['genre'] = df_transformed['genre'].str.strip().str.upper()

# Deduplicate similar publisher names
publisher_mapping = {
    'ELECTRONIC ARTS': 'EA',
    'EA SPORTS': 'EA',
    'SQUARE ENIX': 'SQUARE ENIX',
    'NAMCO BANDAI GAMES': 'BANDAI NAMCO',
    'BANDAI NAMCO GAMES': 'BANDAI NAMCO',
}

df_transformed['publisher'] = df_transformed['publisher'].replace(publisher_mapping)

log_transform(
    "3.1",
    "Text normalization — uppercase, strip whitespace, publisher deduplication",
    "Ensures consistent categorical encoding. Prevents spurious duplicates from formatting variations. Publisher mapping consolidates known aliases.",
)

# Show publisher reduction
unique_publishers_before = df_reduced['publisher'].nunique()
unique_publishers_after = df_transformed['publisher'].nunique()
print(f"   Unique publishers: {unique_publishers_before} → {unique_publishers_after}")

# ─────────────────────────────────────────────────────────────
# 2.3.2 Log Transformation of Sales
# ─────────────────────────────────────────────────────────────

df_transformed['log_sales'] = np.log1p(df_transformed['total_sales'])

# Verify transformation
sales_skew = df_transformed['total_sales'].skew()
log_sales_skew = df_transformed['log_sales'].skew()

log_transform(
    "3.2",
    "Log transformation: log_sales = log1p(total_sales)",
    "Reduces right skewness for parametric modeling. log1p provides numerical stability.",
)

print(f"   Skewness — total_sales: {sales_skew:.3f}, log_sales: {log_sales_skew:.3f}")
print(f"   Skewness reduction: {sales_skew - log_sales_skew:.3f}")

# ─────────────────────────────────────────────────────────────
# 2.3.3 Z-Score / Min-Max Normalization
# ─────────────────────────────────────────────────────────────

# Min-Max scaling for sales features (preserves zero baseline for regional sales)
sales_features = ['total_sales', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales']

minmax_scaler = MinMaxScaler()
df_transformed[[f'{col}_minmax' for col in sales_features]] = minmax_scaler.fit_transform(
    df_transformed[sales_features]
)

# Z-score standardization for critic_score
zscore_scaler = StandardScaler()
df_transformed['critic_score_zscore'] = zscore_scaler.fit_transform(
    df_transformed[['critic_score']]
).flatten()

log_transform(
    "3.3",
    "Normalization — Min-Max for sales, Z-score for critic_score",
    "Min-Max preserves relative proportions and zero baseline (important for regional sales). Z-score standardizes critic_score for fair comparison across models.",
)

# ─────────────────────────────────────────────────────────────
# 2.3.4 Release Date Feature Engineering & Discretization
# ─────────────────────────────────────────────────────────────

# Parse release_date and extract year
df_transformed['release_date'] = pd.to_datetime(df_transformed['release_date'], errors='coerce')
df_transformed['release_year'] = df_transformed['release_date'].dt.year

# Equal-frequency binning of release_year (quantile-based)
n_bins = 5
df_transformed['release_year_bin'] = pd.qcut(
    df_transformed['release_year'],
    q=n_bins,
    labels=[f'era_{i+1}' for i in range(n_bins)],
    duplicates='drop'
)

# Show bin ranges
bin_ranges = df_transformed.groupby('release_year_bin')['release_year'].agg(['min', 'max', 'count'])

log_transform(
    "3.4",
    f"Release date discretization — {n_bins} equal-frequency bins",
    "Converts continuous temporal data into interpretable eras. Equal-frequency ensures balanced representation across time periods for modeling.",
)

print("\n   Release year bins:")
print(bin_ranges.to_string())

# ─────────────────────────────────────────────────────────────
# 2.3.5 Derived Features
# ─────────────────────────────────────────────────────────────

# Regional sales ratio (where total_sales > 0)
for region in ['na', 'jp', 'pal', 'other']:
    df_transformed[f'{region}_ratio'] = np.where(
        df_transformed['total_sales'] > 0,
        df_transformed[f'{region}_sales'] / df_transformed['total_sales'],
        0
    )

# Has regional data flag
df_transformed['has_regional_data'] = (
    (df_transformed[['na_sales', 'jp_sales', 'pal_sales', 'other_sales']].sum(axis=1) > 0)
).astype(int)

# Game count per publisher/developer (brand size proxy)
publisher_counts = df_transformed['publisher'].value_counts()
df_transformed['publisher_game_count'] = df_transformed['publisher'].map(publisher_counts)

developer_counts = df_transformed['developer'].value_counts()
df_transformed['developer_game_count'] = df_transformed['developer'].map(developer_counts)

log_transform(
    "3.5",
    "Feature engineering — regional ratios, brand size proxies, regional data flags",
    "Regional ratios directly address Q2 (genre-regional impact). Publisher/developer game counts proxy brand experience/scale for Q3 (brand effect). Regional data flag enables missingness-aware modeling.",
)

print(f"\n✅ Data transformation complete.")
print(f"📊 Final shape: {df_transformed.shape}")
print(f"📋 Final columns ({len(df_transformed.columns)}):")
for i, col in enumerate(df_transformed.columns, 1):
    print(f"   {i:2d}. {col}")


### 2.4 Data Quality Validation

In [ ]:
# ============================================================
# 2.4 Data Quality Validation
# ============================================================

print("\n" + "━" * 60)
print("M3 — DATA QUALITY VALIDATION")
print("━" * 60)

# Final null check
null_summary = df_transformed.isnull().sum()
null_summary = null_summary[null_summary > 0]

if len(null_summary) > 0:
    print("\n⚠️  Remaining null values:")
    print(null_summary.to_string())
else:
    print("\n✅ No null values in core features.")

# Data type validation
print("\n📋 Data types:")
dtype_summary = pd.DataFrame({
    "Column": df_transformed.columns,
    "Dtype": df_transformed.dtypes.values,
    "Non-Null": df_transformed.notna().sum().values,
    "Unique": [df_transformed[col].nunique() for col in df_transformed.columns]
})
print(dtype_summary.to_string(index=False))

# Range validation for key numeric fields
print("\n📊 Key numeric ranges:")
numeric_cols = ['total_sales', 'log_sales', 'critic_score', 'critic_score_zscore']
range_summary = df_transformed[numeric_cols].agg(['min', 'max', 'mean', 'median', 'std']).round(3)
print(range_summary.to_string())

# Check for duplicates
duplicates = df_transformed.duplicated().sum()
print(f"\n🔄 Duplicate rows: {duplicates}")

print("\n✅ Data quality validation complete.")


### 2.5 Save Cleaned Dataset & Generate Reports

In [ ]:
# ============================================================
# 2.5 Save Cleaned Dataset & Reports
# ============================================================

print("\n" + "━" * 60)
print("SAVING OUTPUTS")
print("━" * 60)

# Create output directories
OUTPUT_DIR = Path("reports/auto_generated")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# Save cleaned dataset
# ─────────────────────────────────────────────────────────────
CLEANED_PATH = "cleaned_vgchartz.csv"
df_transformed.to_csv(CLEANED_PATH, index=False)
print(f"\n💾 Cleaned dataset saved: {CLEANED_PATH}")
print(f"   Shape: {df_transformed.shape}")
print(f"   Size: {Path(CLEANED_PATH).stat().st_size / 1024**2:.2f} MB")
print(f"   Columns: {len(df_transformed.columns)}")

# ─────────────────────────────────────────────────────────────
# Save console-genre aggregation
# ─────────────────────────────────────────────────────────────
AGG_PATH = OUTPUT_DIR / "console_genre_aggregation.csv"
console_genre_agg.to_csv(AGG_PATH, index=False)
print(f"\n💾 Console-Genre aggregation saved: {AGG_PATH}")

print("\n" + "━" * 60)
print("✅ ALL M2 & M3 OUTPUTS SAVED SUCCESSFULLY")
print("━" * 60)
print(f"\n📁 Output files:")
print(f"   ├── {CLEANED_PATH} (cleaned enriched dataset)")
print(f"   └── {AGG_PATH} (console-genre aggregation)")
print(f"\n⚠️  Note: preprocessing.md and summary.md are maintained manually.")


---

## 3. M4 — Exploratory Data Analysis (EDA)

🏗️ **Framework — To be implemented in subsequent iterations.**

### Planned Content

#### 3.1 Nominal Feature Distribution
- Box plots of sales by console, genre, and publisher
- Identification of high-sales combinations (e.g., PS4 × Action)

#### 3.2 Numerical Feature Distribution
- Descriptive statistics for regional sales and log_sales
- Normality tests (Shapiro-Wilk) on log_sales

#### 3.3 Key Visualizations
- Console × Genre sales heatmap (average sales)
- Genre × Region stacked bar chart (sales proportion)
- Top 20 publishers by total sales bar chart

In [ ]:
# ============================================================
# M4 — EDA Framework (Placeholder)
# ============================================================

print("\n" + "━" * 60)
print("M4 — EXPLORATORY DATA ANALYSIS (FRAMEWORK)")
print("━" * 60)

# Load cleaned dataset for EDA
df_eda = pd.read_csv("cleaned_vgchartz.csv")

print(f"\n📊 EDA dataset loaded: {df_eda.shape}")
print("\n🏗️  EDA implementation pending. Planned analyses:")
print("   1. Nominal feature distribution (console, genre, publisher)")
print("   2. Numerical feature distribution (sales, critic_score)")
print("   3. Key visualizations (heatmaps, bar charts, box plots)")

# TODO: Implement EDA visualizations and statistical summaries
# - sns.boxplot(x='console', y='total_sales', data=df_eda)
# - sns.heatmap(console_genre_pivot, annot=True, fmt='.2f')
# - sns.barplot(x='publisher', y='total_sales', data=top_publishers)
# - stats.shapiro(df_eda['log_sales'].sample(5000))

---

## 4. M5 — Correlation Analysis

🏗️ **Framework — To be implemented in subsequent iterations.**

### Planned Content

#### 4.1 Nominal vs Nominal
- Chi-square test for console-genre independence

#### 4.2 Nominal vs Numerical
- ANOVA for platform-genre interaction on log-sales

#### 4.3 Numerical vs Numerical
- Pearson correlation for regional sales
- Variance Inflation Factor (VIF) for multicollinearity detection

In [ ]:
# ============================================================
# M5 — Correlation Analysis Framework (Placeholder)
# ============================================================

print("\n" + "━" * 60)
print("M5 — CORRELATION ANALYSIS (FRAMEWORK)")
print("━" * 60)

print("\n🏗️  Correlation analysis implementation pending. Planned tests:")
print("   1. Chi-square test (console vs genre independence)")
print("   2. ANOVA (platform × genre → log_sales)")
print("   3. Pearson correlation + VIF (regional sales)")

# TODO: Implement correlation analysis
# - chi2_contingency(console_genre_crosstab)
# - f_oneway(*[group['log_sales'].values for name, group in df.groupby(['console', 'genre'])])
# - df[regional_cols].corr() + variance_inflation_factor()

---

## 5. M6 — Model Assistance & Verification

🏗️ **Framework — To be implemented in subsequent iterations.**

### Planned Content

#### 5.1 Random Forest Regression
- Features: platform, genre, publisher, developer (one-hot encoded)
- Target: total_sales
- Output: Feature importances for Q1 and Q3

#### 5.2 XGBoost Regression
- Captures non-linear brand-region interactions
- Validates robustness of Q3 findings

#### 5.3 Cross-Validation & Bootstrap
- 5-fold CV + bootstrap resampling
- Stability criteria: R² ≥ 0.6, Kendall's τ ≥ 0.8

In [ ]:
# ============================================================
# M6 — Model Assistance & Verification Framework (Placeholder)
# ============================================================

print("\n" + "━" * 60)
print("M6 — MODEL ASSISTANCE & VERIFICATION (FRAMEWORK)")
print("━" * 60)

print("\n🏗️  Modeling implementation pending. Planned models:")
print("   1. Random Forest Regression (feature importance for Q1, Q3)")
print("   2. XGBoost Regression (non-linear interactions for Q3)")
print("   3. 5-Fold CV + Bootstrap (R² ≥ 0.6, Kendall's τ ≥ 0.8)")

# TODO: Implement modeling pipeline
# - One-hot encoding for categorical features
# - RandomForestRegressor(n_estimators=200, random_state=42)
# - xgb.XGBRegressor() if XGBOOST_AVAILABLE
# - cross_val_score(model, X, y, cv=5, scoring='r2')
# - Bootstrap confidence intervals for feature importances

print("\n" + "━" * 60)
print("NOTEBOOK EXECUTION COMPLETE")
print("━" * 60)